# syft-bg Integration Tests

Reusable notebook that tests `syft_bg` service management:

1. **Config & Status basics** — init, inspect config, check status
2. **Misconfigured `email_approve`** — missing `gcp_project_id` → graceful error
3. **Start / Stop / Restart** — verify PID lifecycle
4. **Auto-approve job via API** — `auto_approve_job()`, follow-up auto-approval, list/remove rules

### Prerequisites
- `.env` file with `DO_EMAIL`, `DS_EMAIL`, `GCP_PROJECT_ID`, `APP_CREDENTIALS`, `TOKEN_DS`
- OAuth credential files (see Setup cells for first-time token generation)

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup: Environment & Credentials

In [ ]:
import os
from pathlib import Path

for line in Path(".env").read_text().splitlines():
    if line.strip() and not line.startswith("#"):
        key, _, value = line.partition("=")
        os.environ.setdefault(key.strip(), value.strip())

DO_EMAIL = os.environ["DO_EMAIL"]
DS_EMAIL = os.environ["DS_EMAIL"]
GCP_PROJECT_ID = os.environ["GCP_PROJECT_ID"]

APP_CREDENTIALS = Path(os.environ["APP_CREDENTIALS"])
TOKEN_DS = Path(os.environ["TOKEN_DS"])
DO_CREDENTIALS = Path("/Users/bitsofsteve/code/syft-client/creds.stephen.json")
DO_TOKEN = Path("/Users/bitsofsteve/code/syft-client/notebooks/beach/internal/token_do.json")

assert APP_CREDENTIALS.exists(), f"App credentials missing: {APP_CREDENTIALS}"
assert DO_CREDENTIALS.exists(), f"DO credentials missing: {DO_CREDENTIALS}"

print(f"DO: {DO_EMAIL}")
print(f"DS: {DS_EMAIL}")

### First-time token generation

Uncomment and run these cells once to generate OAuth tokens. Each opens a browser for consent.

In [ ]:
import syft_client as sc

# --- DS token (Drive scopes only) ---
# sc.credentials_to_token(
#     credentials_path=str(APP_CREDENTIALS),
#     output_path=str(TOKEN_DS),
# )
# print(f"DS token saved to: {TOKEN_DS}")

In [ ]:
# --- DO token (Drive + Gmail + PubSub scopes) ---
# sc.credentials_to_token(
#     credentials_path=str(DO_CREDENTIALS),
#     output_path=str(DO_TOKEN),
#     do_scopes=True,
# )
# print(f"DO token saved to: {DO_TOKEN}")

In [ ]:
assert TOKEN_DS.exists(), f"DS token missing: {TOKEN_DS} — uncomment generation cell above"
assert DO_TOKEN.exists(), f"DO token missing: {DO_TOKEN} — uncomment generation cell above"
print("All tokens ready")

## Setup: Login & Peers

In [ ]:
import syft_client as sc

do_client = sc.login_do(email=DO_EMAIL, token_path=str(DO_TOKEN))
ds_client = sc.login_ds(email=DS_EMAIL, token_path=str(TOKEN_DS))

In [ ]:
ds_client.add_peer(DO_EMAIL)
do_client.load_peers()
do_client.approve_peer_request(DS_EMAIL, peer_must_exist=False)
print("Peer relationship established")

## Setup: Test Dataset

In [ ]:
try:
    do_client.create_dataset(
        name="testdata",
        mock_path="data/mock.txt",
        private_path="data/private.txt",
        users=[DS_EMAIL],
    )
except FileExistsError:
    print("Dataset 'testdata' already exists, skipping creation")

---
## Test 1: Config & Status Basics

In [ ]:
import syft_bg

syft_bg.reset()

In [ ]:
syft_bg.init(
    do_email=DO_EMAIL,
    token_path=str(DO_TOKEN),
    settings={
        "email_approve": {"gcp_project_id": GCP_PROJECT_ID},
    },
)

In [ ]:
# Inspect config — should show do_email, token paths, and gcp_project_id
syft_bg.config

In [ ]:
# Status before starting any services — all should be stopped
status = syft_bg.status
print(status)

for name, info in status.service_infos.items():
    assert info.status.value in ("stopped", "unknown"), f"{name} should be stopped, got {info.status}"
print("\nAll services stopped as expected")

---
## Test 2: Misconfigured `email_approve` (no `gcp_project_id`)

In [ ]:
# Re-init WITHOUT gcp_project_id for email_approve
syft_bg.reset()
syft_bg.init(
    do_email=DO_EMAIL,
    token_path=str(DO_TOKEN),
    # intentionally no email_approve settings
)

In [ ]:
import time

# Start only email_approve — it should fail during setup
syft_bg.ensure_running(services=["email_approve"])

# Give the subprocess time to attempt setup and write error state
time.sleep(5)

In [ ]:
from syft_bg.services.base import ServiceStatus

status = syft_bg.status
print(status)

ea_info = status.service_infos["email_approve"]
assert ea_info.status == ServiceStatus.ERROR, (
    f"Expected email_approve to be in ERROR state, got {ea_info.status}"
)
print(f"\nemail_approve correctly in ERROR state")
print(f"Error: {ea_info.error[:200] if ea_info.error else 'no error message'}...")

In [ ]:
# Check logs confirm the gcp_project_id error
log_lines = syft_bg.logs("email_approve", n=30, as_list=True)
error_lines = [l for l in log_lines if "gcp_project_id" in l.lower() or "GCP project ID" in l]
assert error_lines, "Expected gcp_project_id error in email_approve logs"
print("Logs confirm missing gcp_project_id:")
for line in error_lines:
    print(f"  {line}")

---
## Test 3: Start / Stop / Restart Services

In [ ]:
# Clean slate with proper config
syft_bg.reset()
syft_bg.init(
    do_email=DO_EMAIL,
    token_path=str(DO_TOKEN),
    settings={
        "email_approve": {"gcp_project_id": GCP_PROJECT_ID},
    },
)

In [ ]:
# Start sync, notify, approve (skip email_approve — it needs Pub/Sub infra)
# Start sync first so it creates the cache directory before approve needs it
SERVICES = ["sync", "notify", "approve"]

syft_bg.ensure_running(services=["sync"], restart=True)
time.sleep(10)  # let sync create .cache dir

syft_bg.ensure_running(services=["notify", "approve"], restart=True)
time.sleep(5)

status = syft_bg.status
print(status)

In [ ]:
# Record initial PIDs
initial_pids = {}
for svc in SERVICES:
    info = status.service_infos[svc]
    assert info.status in (ServiceStatus.RUNNING, ServiceStatus.STARTING), (
        f"{svc} should be running, got {info.status}"
    )
    initial_pids[svc] = info.pid
    print(f"{svc}: PID {info.pid} ({info.status.value})")

print(f"\nAll {len(SERVICES)} services running")

In [ ]:
# Restart the approve service — PID should change
syft_bg.restart("approve")
time.sleep(3)

new_status = syft_bg.status
new_approve_info = new_status.service_infos["approve"]
assert new_approve_info.pid != initial_pids["approve"], (
    f"approve PID should have changed: was {initial_pids['approve']}, still {new_approve_info.pid}"
)
print(f"approve restarted: PID {initial_pids['approve']} -> {new_approve_info.pid}")

In [ ]:
# Stop approve
syft_bg.stop("approve")
time.sleep(1)

stopped_status = syft_bg.status
assert stopped_status.service_infos["approve"].status == ServiceStatus.STOPPED, (
    f"approve should be stopped, got {stopped_status.service_infos['approve'].status}"
)
print("approve stopped")

# Other services should still be running
for svc in ["sync", "notify"]:
    assert stopped_status.service_infos[svc].status in (ServiceStatus.RUNNING, ServiceStatus.STARTING), (
        f"{svc} should still be running"
    )
print("sync and notify still running")

In [ ]:
# Start approve again
syft_bg.start("approve")
time.sleep(3)

restarted_status = syft_bg.status
assert restarted_status.service_infos["approve"].status in (ServiceStatus.RUNNING, ServiceStatus.STARTING), (
    f"approve should be running again, got {restarted_status.service_infos['approve'].status}"
)
print(f"approve running again: PID {restarted_status.service_infos['approve'].pid}")
print("\nStart/Stop/Restart test passed")

---
## Test 4: Auto-Approve Job via API

DS submits a parameterized job. DO uses `syft_bg.auto_approve_job()` to create an
auto-approval rule. The `approve` service picks it up, runs the job, and auto-approves
a follow-up job with the same code but different params.

In [ ]:
import json
import tempfile
import uuid

def create_parameterized_job(main_file: str, params: dict) -> Path:
    """Create a temp project dir with a script and params.json."""
    project_dir = Path(tempfile.mkdtemp(prefix="syftbg_test_"))
    (project_dir / main_file).write_text(
        'import os, json\n'
        '\n'
        'with open("params.json") as f:\n'
        '    params = json.load(f)\n'
        '\n'
        'os.makedirs("outputs", exist_ok=True)\n'
        'with open("outputs/result.json", "w") as f:\n'
        '    json.dump({"params": params, "status": "done"}, f)\n'
        '\n'
        'print(f"Result: {params}")\n'
    )
    (project_dir / "params.json").write_text(json.dumps(params))
    return project_dir

In [ ]:
# Ensure services are running for this test
syft_bg.ensure_running(services=["sync"], restart=True)
time.sleep(10)
syft_bg.ensure_running(services=["notify", "approve"], restart=True)
time.sleep(5)
print(syft_bg.status)

### 4a: Submit a job and create auto-approval rule

In [ ]:
JOB_NAME = f"api_auto_{uuid.uuid4().hex[:8]}"
job_dir = create_parameterized_job(main_file="main.py", params={"run": 1})

ds_client.submit_python_job(
    user=DO_EMAIL,
    code_path=str(job_dir),
    job_name=JOB_NAME,
    entrypoint="main.py",
    force_submission=True,
)
print(f"Submitted: {JOB_NAME}")

In [ ]:
# Wait for sync service to make the job visible on DO side
for attempt in range(12):
    pending = [j for j in do_client.jobs if j.name == JOB_NAME]
    if pending:
        break
    print(f"  waiting for job to appear on DO side... ({attempt+1})")
    time.sleep(5)

assert pending, f"DO never saw job {JOB_NAME}"
assert pending[0].status == "pending", f"Expected pending, got {pending[0].status}"
print(f"DO sees job: {JOB_NAME} (status: {pending[0].status})")

In [ ]:
# Create auto-approval rule from the pending job
result = syft_bg.auto_approve_job(pending[0], file_paths=["params.json"])
assert result.success, f"auto_approve_job failed: {result.error}"
print(result)

### 4b: Verify auto-approval config

In [ ]:
rules = syft_bg.list_auto_approvals()
assert JOB_NAME in rules, f"Rule '{JOB_NAME}' not found in auto-approvals"

rule = rules[JOB_NAME]
content_paths = {e.relative_path for e in rule.file_contents}
assert content_paths == {"main.py"}, f"Expected content match on main.py, got {content_paths}"
assert "params.json" in rule.file_paths, "params.json should be name-only matched"
assert DS_EMAIL in rule.peers, f"DS email should be in peers, got {rule.peers}"

print(f"Rule '{JOB_NAME}':")
print(f"  content-matched: {content_paths}")
print(f"  name-matched:    {rule.file_paths}")
print(f"  peers:           {rule.peers}")

In [ ]:
# Also visible in status
status = syft_bg.status
assert JOB_NAME in status.auto_approvals, "Rule should appear in syft_bg.status"
print("Rule visible in syft_bg.status")

### 4c: Wait for the approve service to run the job

In [ ]:
# The approve service polls every 5s — wait for it to pick up the job
for attempt in range(24):
    job = next((j for j in do_client.jobs if j.name == JOB_NAME), None)
    if job and job.status == "done":
        break
    print(f"  job status: {job.status if job else 'not found'} ({attempt+1})")
    time.sleep(5)

assert job and job.status == "done", f"Job never completed, status: {job.status if job else 'missing'}"
print(f"Job {JOB_NAME}: {job.status}")

In [ ]:
# DS fetches result
ds_job = next((j for j in ds_client.jobs if j.name == JOB_NAME), None)
assert ds_job, f"DS can't see job {JOB_NAME}"
assert ds_job.status == "done", f"DS sees status {ds_job.status}"

result_data = json.loads(ds_job.output_paths[0].read_text())
print(f"DS received result: {result_data}")
assert result_data["params"]["run"] == 1

### 4d: Follow-up job — should be auto-approved (no manual intervention)

In [ ]:
FOLLOWUP_NAME = f"api_auto_{uuid.uuid4().hex[:8]}"
followup_dir = create_parameterized_job(main_file="main.py", params={"run": 2})

ds_client.submit_python_job(
    user=DO_EMAIL,
    code_path=str(followup_dir),
    job_name=FOLLOWUP_NAME,
    entrypoint="main.py",
    force_submission=True,
)
print(f"Submitted follow-up: {FOLLOWUP_NAME}")

In [ ]:
# Wait for auto-approval + execution
for attempt in range(24):
    job2 = next((j for j in do_client.jobs if j.name == FOLLOWUP_NAME), None)
    if job2 and job2.status == "done":
        break
    print(f"  follow-up status: {job2.status if job2 else 'not found'} ({attempt+1})")
    time.sleep(5)

assert job2 and job2.status == "done", (
    f"Follow-up job not auto-approved/completed, status: {job2.status if job2 else 'missing'}"
)
print(f"Follow-up {FOLLOWUP_NAME}: {job2.status} (auto-approved!)")

In [ ]:
# DS fetches follow-up result
ds_job2 = next((j for j in ds_client.jobs if j.name == FOLLOWUP_NAME), None)
assert ds_job2 and ds_job2.status == "done"

result_data2 = json.loads(ds_job2.output_paths[0].read_text())
print(f"DS received follow-up result: {result_data2}")
assert result_data2["params"]["run"] == 2

### 4e: Remove auto-approval rule

In [ ]:
remove_result = syft_bg.remove_auto_approve(JOB_NAME)
assert remove_result.success, f"Failed to remove: {remove_result.error}"

rules_after = syft_bg.list_auto_approvals()
assert JOB_NAME not in rules_after, f"Rule '{JOB_NAME}' should have been removed"
print(f"Rule '{JOB_NAME}' removed")
print(f"Remaining rules: {list(rules_after.keys()) or '(none)'}")

---
## Debugging: Service Logs

In [ ]:
for svc in ("sync", "notify", "approve"):
    print(f"\n{'='*20} {svc} {'='*20}")
    syft_bg.logs(svc, n=20)

## Final status

In [ ]:
syft_bg.status

---
## Cleanup

In [ ]:
syft_bg.stop()

In [ ]:
# Uncomment to fully reset all syft-bg state
# syft_bg.reset()